In [2]:
sentence = "Life is short, eat desert first"

dc = { s:i for i,s in enumerate(sorted(sentence.replace(',', ' ').split()))}
dc

{'Life': 0, 'desert': 1, 'eat': 2, 'first': 3, 'is': 4, 'short': 5}

In [3]:
import torch

sentence_int = torch.tensor([dc[s] for s in sentence.replace(',', ' ').split()])
sentence_int

tensor([0, 4, 5, 2, 1, 3])

In [4]:
vocab_size = 50_000

torch.manual_seed(123)
embed = torch.nn.Embedding(vocab_size, 3)
embedded_sentence = embed(sentence_int).detach()

print(embedded_sentence)
print(embedded_sentence.shape)

tensor([[ 0.3374, -0.1778, -0.3035],
        [ 0.1794,  1.8951,  0.4954],
        [ 0.2692, -0.0770, -1.0205],
        [-0.2196, -0.3792,  0.7671],
        [-0.5880,  0.3486,  0.6603],
        [-1.1925,  0.6984, -1.4097]])
torch.Size([6, 3])


In [5]:
d = embedded_sentence.shape[1] # embedding_dim

d_q, d_k, d_v = 2, 2, 4

W_q = torch.nn.Parameter(torch.rand(d, d_q))
W_k = torch.nn.Parameter(torch.rand(d , d_k))
w_V = torch.nn.Parameter(torch.rand(d, d_v))

In [6]:
x = embedded_sentence

In [7]:
queries = x @ W_q
keys = x @ W_k
values = x @ w_V

print(f"Query shape: ", queries.shape)
print(f"Key shape: ", keys.shape)
print(f"Value shape: ", values.shape)

Query shape:  torch.Size([6, 2])
Key shape:  torch.Size([6, 2])
Value shape:  torch.Size([6, 4])


In [8]:
attention_scores = queries @ keys.T

attention_weights = torch.softmax(attention_scores / d_k ** 0.5, dim=0)

In [9]:
context_vectors = attention_weights @ values

context_vectors

tensor([[-0.3275, -0.1044,  0.0592, -0.0750],
        [ 0.2912,  0.1914,  0.8192,  0.7015],
        [-0.4094, -0.1476,  0.0250, -0.1116],
        [-0.3308, -0.0812,  0.0719, -0.1020],
        [-0.2476, -0.0625,  0.1117, -0.0216],
        [-0.5597, -0.2133, -0.0040, -0.1694]], grad_fn=<MmBackward0>)

In [10]:
import torch.nn as nn

class SelfAttention(nn.Module):

    def __init__(self, d_in, d_out_kq, d_out_v):
        super().__init__()

        self.d_out_kq = d_out_kq

        self.W_q = nn.Parameter(torch.rand(d_in, d_out_kq))
        self.W_k = nn.Parameter(torch.rand(d_in, d_out_kq))
        self.W_v = nn.Parameter(torch.rand(d_in, d_out_v))

    def forward(self, x):
        queries = x @ self.W_q
        keys = x @ self.W_k
        values = x @ self.W_v

        attention_scores = queries @ keys.T
        attention_weights = torch.softmax(attention_scores / self.d_out_kq ** 0.5, dim=-1)

        context_vec = attention_weights @ values
        return context_vec



In [11]:
torch.manual_seed(123)

sa = SelfAttention(d_in=3, d_out_kq=2, d_out_v=4)

print(sa(embedded_sentence))

tensor([[-0.1564,  0.1028, -0.0763, -0.0764],
        [ 0.5313,  1.3607,  0.7891,  1.3110],
        [-0.3542, -0.1234, -0.2626, -0.3706],
        [ 0.0071,  0.3345,  0.0969,  0.1998],
        [ 0.1008,  0.4780,  0.2021,  0.3674],
        [-0.5296, -0.2799, -0.4107, -0.6006]], grad_fn=<MmBackward0>)
